# Adaptive affine hedging: three paired-seed sweeps

Run with a **local Python kernel in VS Code**. This notebook uses your existing `awdro` project, but embeds the full-coupling fast solver from your attached `fast_solvers(1).py`; it never imports a neighboring-path variant or overwrites your project files. Edit `LOCAL_REPO` below if needed. Run the cells in order.

**The frozen-holdings correction changes the hedge representation.** A discrete history table has no specified derivative away from its nodes. All three methods here use the same differentiable affine policy, including stock and variance responses. These results therefore replace, rather than reproduce, the old unrestricted history-table hedge experiment.

**The third curve is a first-order ordinary-Wasserstein model, not a causal solver.** It linearizes trading gains with the full policy derivative and retains the call/shortfall kinks. This is an intentional change from the old plug-in `nominal + budget × gradient norm` at an optimized kink. It is not an exact robust Heston price, and no ordering between this curve and the causal curves is asserted.

Figures show seed means, solid curves, faint horizontal/vertical gridlines, and a legend. No figure titles or annotations are added. Color mapping: **purple = nonrectangular causal; apricot = rectangular hull; lilac = first-order ordinary W2 model**. Full per-seed results are saved for audit. Additional cells below the original call experiment repeat all three sweeps for eight more option payoffs.

Results, policies, configuration and PDF/PNG figures save automatically under `~/awdro_results/affine_three_sweeps_<hash>/`. On a local kernel this is your computer; a Colab/remote kernel would save to that remote machine instead. Run completion checkpoints allow safe resume. No failed seeds are silently dropped.


In [ ]:
# Uncomment only if packages are missing, then restart the kernel.
# %pip install numpy pandas scipy torch cvxpy clarabel matplotlib

from pathlib import Path
import sys
import types
import inspect
import importlib.metadata
import hashlib
import json
import time

LOCAL_REPO = Path('/Users/jacve/Git/awdro')  # edit if your project moved
folders = ['hedging', 'path simulation', 'inner maximization', 'outer minimization']
search_roots = [LOCAL_REPO, LOCAL_REPO / 'awdro', Path.cwd(), *Path.cwd().parents]
root = next((p for p in search_roots if all((p / f).is_dir() for f in folders)), None)
if root is None:
    raise FileNotFoundError('Set LOCAL_REPO to the folder containing hedging/ and path simulation/.')
for p in [root, *(root / f for f in folders)]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import scipy
from scipy.sparse import csr_matrix, hstack, vstack
import torch
import cvxpy as cp
import matplotlib.pyplot as plt
from Heston import Heston
from SuperhedgingProblem import SuperhedgingProblem
from RectangularSolver import RectangularSolver
from RelaxedSuperhedgingSolver import RelaxedSuperhedgingSolver
if 'CLARABEL' not in cp.installed_solvers():
    raise RuntimeError('Install clarabel and restart the kernel.')


## Model and mathematical scope

For $t=0,\ldots,T-1$ the same coefficient vector is optimized jointly with capital in every method:
$$H_t(S_t,V_t)=a_t+b_t(S_t/S_0-1)+c_t(V_t-V_0),\qquad
G_\theta(z)=\sum_t H_t(z_t)(S_{t+1}-S_t).$$
The default coefficient bounds are $[-2,2]$, actual holdings are constrained to $[-2,2]$ on all finite candidate states, and $b_0=c_0=0$. This is a restricted adaptive Markov family, not all predictable hedges. The extension outside the finite support is affine, so its holdings are not globally bounded.

The objective is $x+\kappa\sup_Q E_Q[( (S_T-K)_+-x-G_\theta)_+]$. The finite causal coupling cost is
$$c(z,y)=\sum_{t=1}^T[(S_t-Y^S_t)^2+(V_t-Y^V_t)^2],\qquad E[c]\le\varepsilon^2.$$
Both coordinates can move; the initial state is fixed. The budget in the configuration is $\varepsilon$, not $\varepsilon^2$. These are raw-coordinate units; stock and variance are not normalized, and there is no $\Delta t$ factor. Keeping this convention reproduces the previous cost, but increasing the number of dates changes this path metric. The time-step sweep is therefore **not** by itself a convergence study at a fixed continuous-time transport metric.

The reference is a sampled Heston tree. The adversarial support is the full product of its date-wise states, with every reference/adversarial pair allowed. The rectangular calculation pastes conditional laws of the finite causal ambiguity set along adversarial histories. It is the history-rectangular hull of that finite set; calling it state-rectangular requires taking the full history as state. Restricting the hedge to affine Markov policies does not change this definition. These are finite-model numerical values, not a proof of convergence to continuous-state Jiang values.

For the third method, with reference atom $z_i$, write $d_i=\nabla_zG_\theta(z_i)$, including derivatives of $H_t$. Keep the loss branches
$$\max\{0,\ a_{i0}+g_{i0}^{\top}\delta,\ a_{i1}+g_{i1}^{\top}\delta\},$$
where $a_{i0}=-x-G_\theta(z_i)$, $a_{i1}=S_{iT}-K+a_{i0}$, $g_{i0}=-d_i$, and $g_{i1}=e_{S_T}-d_i$.
Its ordinary-W2 robust expectation is solved by
$$\inf_{\lambda\ge0,z_i\ge0}\lambda\varepsilon^2+\sum_i p_i z_i,
\qquad 4\lambda(z_i-a_{ik})\ge\|g_{ik}\|_2^2,\quad k=0,1.$$
These are second-order cone constraints; reference atoms may split mass. At zero budget use the nominal problem directly. This avoids the zero-gradient artifact at a shortfall/call kink.

For this affine hedge, trading gains are quadratic in path coordinates. For a fixed policy, $G_\theta(z+\delta)-G_\theta(z)-\nabla G_\theta(z)^\top\delta=\delta^\top Q_\theta\delta$. Thus the fixed-policy linearization error in the penalized **unrestricted ordinary-W2** problem is at most $\kappa\|Q_\theta\|_{op}\varepsilon^2$. The notebook records that bound. It is not an error bar for the optimized causal value or for its finite-support discretization. Ordinary perturbations here range over all real post-initial coordinates, so positivity is not imposed on their stock/variance endpoints. Do not describe the third curve as using the same support or causal ambiguity set as the other two.


In [ ]:
# The only change to the attached fast LP source is dispatch through
# self._solve_lp, allowing the common affine holding constraints to be added.
FAST_SOURCE = '"""Drop-in faster versions of RectangularSolver and RelaxedSuperhedgingSolver.\n\nNothing in the original files is modified.  The subclasses below build exactly\nthe same matrices / linear programs, but\n\n* the setup (adversarial paths, transport costs, causality constraints) is\n  vectorised with numpy instead of Python loops, and\n* the two adaptive-hedge LPs are assembled directly as scipy sparse matrices\n  and solved by HiGHS through scipy.optimize.linprog, instead of building\n  one cvxpy expression per path / per history.\n\nUsage in the notebook (everything else stays the same):\n\n    from fast_solvers import FastRectangularSolver, FastRelaxedSuperhedgingSolver\n    rectangular = FastRectangularSolver(model, times, BUDGET, discretization)\n    outer = FastRelaxedSuperhedgingSolver(problem=problem,\n                                          inner_solver=nonrectangular,\n                                          penalty=PENALTY)\n"""\n\nimport numpy as np\nimport torch\nfrom scipy.optimize import linprog\nfrom scipy.sparse import csr_matrix, coo_matrix, hstack, vstack, identity\n\nfrom RectangularSolver import RectangularSolver\nfrom RelaxedSuperhedgingSolver import RelaxedSuperhedgingSolver\n\n\n# --------------------------------------------------------------------------\n# Setup: same outputs as RectangularSolver, without Python loops\n# --------------------------------------------------------------------------\nclass FastRectangularSolver(RectangularSolver):\n\n    def _build_adversarial_paths(self):\n        initial_state = self.reference_nodes[0]["state"]\n        candidate_sets = [\n            self.adversarial_candidates[t]\n            for t in range(1, len(self.trading_times))\n        ]\n        sizes = [len(c) for c in candidate_sets]\n        m = int(np.prod(sizes))\n\n        # C-order index grid: last date varies fastest, like itertools.product.\n        index = np.indices(sizes).reshape(len(sizes), -1)\n\n        paths = np.empty((m, len(sizes) + 1, 2), dtype=float)\n        paths[:, 0, :] = initial_state\n        for k, candidates in enumerate(candidate_sets):\n            paths[:, k + 1, :] = candidates[index[k]]\n        return paths\n\n    def _build_transport_costs(self):\n        reference = self.reference_paths[:, 1:, :]\n        adversarial = self.adversarial_paths[:, 1:, :]\n        costs = np.empty((reference.shape[0], adversarial.shape[0]))\n        for i in range(reference.shape[0]):\n            diff = adversarial - reference[i][None, :, :]\n            costs[i] = np.einsum("jtk,jtk->j", diff, diff)\n        return costs\n\n    def _build_causality_constraints(self):\n        n = self.reference_paths.shape[0]\n        m = self.adversarial_paths.shape[0]\n        probabilities = self.reference_probabilities\n\n        if not np.isfinite(probabilities).all() or np.any(probabilities <= 0):\n            raise ValueError(\n                "Reference-path probabilities must be finite and positive."\n            )\n\n        rows, cols, vals = [], [], []\n        next_row = 0\n        column_offsets = np.arange(m)\n\n        for t in range(1, len(self.trading_times) - 1):\n            reference_prefixes = self.reference_paths[:, :t + 1, :].reshape(n, -1)\n            adversarial_prefixes = self.adversarial_paths[:, :t + 1, :].reshape(m, -1)\n\n            _, reference_groups = np.unique(\n                reference_prefixes, axis=0, return_inverse=True\n            )\n            _, adversarial_groups = np.unique(\n                adversarial_prefixes, axis=0, return_inverse=True\n            )\n            reference_groups = np.asarray(reference_groups).ravel()\n            adversarial_groups = np.asarray(adversarial_groups).ravel()\n            n_adv_groups = int(adversarial_groups.max()) + 1\n\n            members, anchors = [], []\n            for group in range(int(reference_groups.max()) + 1):\n                indices = np.flatnonzero(reference_groups == group)\n                members.extend(indices[1:].tolist())\n                anchors.extend([int(indices[0])] * (len(indices) - 1))\n\n            members = np.asarray(members, dtype=int)\n            anchors = np.asarray(anchors, dtype=int)\n            if members.size == 0:\n                continue\n\n            pair = np.arange(members.size)\n            row_block = (\n                next_row\n                + pair[:, None] * n_adv_groups\n                + adversarial_groups[None, :]\n            )\n\n            rows += [row_block.ravel(), row_block.ravel()]\n            cols += [\n                (members[:, None] * m + column_offsets[None, :]).ravel(),\n                (anchors[:, None] * m + column_offsets[None, :]).ravel(),\n            ]\n            vals += [\n                np.repeat(1.0 / probabilities[members], m),\n                np.repeat(-1.0 / probabilities[anchors], m),\n            ]\n            next_row += members.size * n_adv_groups\n\n        if rows:\n            A_eq = coo_matrix(\n                (np.concatenate(vals),\n                 (np.concatenate(rows), np.concatenate(cols))),\n                shape=(next_row, n * m),\n                dtype=float,\n            ).tocsr()\n        else:\n            A_eq = csr_matrix((0, n * m), dtype=float)\n\n        return A_eq, np.zeros(next_row, dtype=float)\n\n\n# --------------------------------------------------------------------------\n# Outer problems: same LPs, assembled as sparse matrices, solved by HiGHS\n# --------------------------------------------------------------------------\ndef _solve_lp(c, A_ub, b_ub, lower, upper, label):\n    bounds = np.column_stack([lower, upper])\n    result = linprog(\n        c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs"\n    )\n    if result.status != 0 or result.x is None:\n        raise RuntimeError(f"{label} LP failed: {result.message}")\n    return result\n\n\nclass FastRelaxedSuperhedgingSolver(RelaxedSuperhedgingSolver):\n    _solve_lp = staticmethod(_solve_lp)\n\n    # Refuse LPs with more constraint rows than this, so an infeasible grid\n    # fails immediately with a clear message instead of hanging or exhausting\n    # memory.  Raise it on a large machine.\n    max_lp_rows = 5_000_000\n\n    def _check_size(self, rows, label):\n        if rows > self.max_lp_rows:\n            raise RuntimeError(\n                f"{label} LP would have {rows:,} rows "\n                f"(limit {self.max_lp_rows:,}); skipped."\n            )\n\n    # ---- shared preparation -------------------------------------------\n    def _prepare(self, transport_model, capital_bounds, holding_bounds):\n        bounds = np.asarray([capital_bounds, holding_bounds], dtype=float)\n        if bounds.shape != (2, 2) or not np.isfinite(bounds).all():\n            raise ValueError("Provide two finite bounds for capital and holding.")\n        if np.any(bounds[:, 0] >= bounds[:, 1]):\n            raise ValueError("Each lower bound must be below its upper bound.")\n\n        paths = transport_model.adversarial_paths\n        n_paths = paths.shape[0]\n\n        with torch.no_grad():\n            payoff_tensor = self.problem.payoffs(\n                torch.tensor(paths, dtype=torch.float64)\n            )\n        payoffs = payoff_tensor.detach().cpu().numpy()\n        if payoffs.shape != (n_paths,) or not np.isfinite(payoffs).all():\n            raise ValueError("Provide one finite payoff per adversarial path.")\n\n        G, histories_by_time, offsets = self._build_adaptive_gains_matrix(paths)\n        return bounds, paths, payoffs, G, histories_by_time, offsets\n\n    @staticmethod\n    def _result(x, nh, m, histories_by_time, offsets, value, shortfall):\n        return {\n            "capital": float(x[0]),\n            "holdings": np.asarray(x[1:1 + nh], dtype=float).copy(),\n            "histories_by_time": histories_by_time,\n            "offsets": offsets,\n            "value": float(value),\n            "worst_case_shortfall": float(shortfall),\n        }\n\n    # ---- nonrectangular -----------------------------------------------\n    def solve_adaptive_hedge_nonrectangular(\n        self, transport_model, capital_bounds, holding_bounds\n    ):\n        bounds, paths, payoffs, G, histories_by_time, offsets = (\n            self._prepare(transport_model, capital_bounds, holding_bounds)\n        )\n        m = paths.shape[0]\n        nh = G.shape[1]\n        n = transport_model.reference_paths.shape[0]\n        A_eq, b_eq = transport_model.A_eq, transport_model.b_eq\n        A_ub, b_ub = transport_model.A_ub, transport_model.b_ub\n        R, Rb = A_eq.shape[0], A_ub.shape[0]\n        self._check_size(n * m + m, "Nonrectangular")\n\n        i_h = 1\n        i_s = 1 + nh\n        i_mu = i_s + m\n        i_beta = i_mu + R\n        n_vars = i_beta + Rb\n\n        c = np.zeros(n_vars)\n        c[0] = 1.0\n        c[i_mu:i_mu + R] = self.penalty * b_eq\n        c[i_beta:] = self.penalty * b_ub\n\n        # shortfall + capital + G @ holdings >= payoff\n        block_gain = hstack(\n            [\n                csr_matrix(-np.ones((m, 1))),\n                -G,\n                -identity(m, format="csr"),\n                csr_matrix((m, R)),\n                csr_matrix((m, Rb)),\n            ],\n            format="csr",\n        )\n\n        # A_eq.T @ mu + A_ub.T @ beta >= shortfall repeated for each reference path\n        repeat = csr_matrix(\n            (np.ones(n * m), (np.arange(n * m), np.tile(np.arange(m), n))),\n            shape=(n * m, m),\n        )\n        block_dual = hstack(\n            [csr_matrix((n * m, 1 + nh)), repeat, -A_eq.T, -A_ub.T],\n            format="csr",\n        )\n\n        A = vstack([block_gain, block_dual], format="csr")\n        b = np.concatenate([-payoffs, np.zeros(n * m)])\n\n        lower = np.concatenate([\n            [bounds[0, 0]], np.full(nh, bounds[1, 0]),\n            np.zeros(m), np.full(R, -np.inf), np.zeros(Rb),\n        ])\n        upper = np.concatenate([\n            [bounds[0, 1]], np.full(nh, bounds[1, 1]),\n            np.full(m, np.inf), np.full(R, np.inf), np.full(Rb, np.inf),\n        ])\n\n        res = self._solve_lp(c, A, b, lower, upper, "Nonrectangular")\n        x = res.x\n        shortfall = b_eq @ x[i_mu:i_mu + R] + b_ub @ x[i_beta:]\n        return self._result(\n            x, nh, m, histories_by_time, offsets, res.fun, shortfall\n        )\n\n    # ---- rectangular --------------------------------------------------\n    def solve_adaptive_hedge_rectangular(\n        self, transport_model, capital_bounds, holding_bounds\n    ):\n        bounds, paths, payoffs, G, histories_by_time, offsets = (\n            self._prepare(transport_model, capital_bounds, holding_bounds)\n        )\n        m, n_times, _ = paths.shape\n        nh = G.shape[1]\n        n = transport_model.reference_paths.shape[0]\n        i_s = 1 + nh\n\n        block_gain = hstack(\n            [csr_matrix(-np.ones((m, 1))), -G, -identity(m, format="csr")],\n            format="csr",\n        )\n\n        if transport_model.budget == 0:\n            return self._rectangular_zero_budget(\n                transport_model, bounds, paths, payoffs, block_gain,\n                nh, m, histories_by_time, offsets,\n            )\n\n        A_eq, b_eq = transport_model.A_eq, transport_model.b_eq\n        A_ub, b_ub = transport_model.A_ub, transport_model.b_ub\n        if A_ub.shape[0] != 1:\n            raise NotImplementedError("Expected a single global budget row.")\n        R = A_eq.shape[0]\n        nm = n * m\n\n        # Prefix groups at each date (history of length t + 1).\n        group_id, n_groups = [], []\n        for t in range(n_times - 1):\n            prefixes = paths[:, :t + 1, :].reshape(m, -1)\n            _, g = np.unique(prefixes, axis=0, return_inverse=True)\n            g = np.asarray(g).ravel()\n            group_id.append(g)\n            n_groups.append(int(g.max()) + 1)\n\n        # Number of histories (prefixes) over all dates, before building anything.\n        n_prefix_total = 0\n        for t in range(n_times - 1):\n            prefixes = paths[:, :t + 1, :].reshape(m, -1)\n            n_prefix_total += len(np.unique(prefixes, axis=0))\n        self._check_size(n_prefix_total * (nm + 1), "Rectangular")\n\n        # Every history in the candidate space is reachable when budget > 0\n        # (a vanishing sliver of mass can always be sent to it), so the\n        # per-history reachability LP of the original code is skipped.\n        block = R + 3                       # e (R + 1), b (1), v (1)\n        group_start = []\n        position = i_s + m\n        prefix_index = {}\n        count = 0\n        for t in range(n_times - 1):\n            starts = []\n            for g in range(n_groups[t]):\n                starts.append(position)\n                prefix_index[(t, g)] = count\n                position += block\n                count += 1\n            group_start.append(np.asarray(starts))\n        n_vars = position\n        n_prefix = count\n\n        def child_variable(t, j):\n            """Variable holding the value of path j\'s prefix of length t + 2."""\n            if t + 1 == n_times - 1:\n                return i_s + j\n            return group_start[t + 1][group_id[t + 1][j]] + R + 2\n\n        # (nm + 1) x R block shared by every prefix: -base_eq^T\n        Bt = vstack(\n            [-A_eq.T, csr_matrix(b_eq.reshape(1, -1))], format="coo"\n        )\n        ub_column = np.concatenate([-A_ub.toarray().ravel(), [b_ub[0]]])\n        ub_rows = np.flatnonzero(ub_column)\n        ub_vals = ub_column[ub_rows]\n\n        rows, cols, vals = [], [], []\n        v_rows, v_cols, v_vals = [], [], []\n        i_range = np.arange(n)[:, None] * m\n\n        for t in range(n_times - 1):\n            for g in range(n_groups[t]):\n                p = prefix_index[(t, g)]\n                start = group_start[t][g]\n                row0 = p * (nm + 1)\n\n                rows.append(Bt.row + row0)\n                cols.append(Bt.col + start)\n                vals.append(Bt.data)\n\n                J = np.flatnonzero(group_id[t] == g)\n\n                # normalisation dual: -1 on columns (i, j) with j in history\n                norm_rows = (row0 + i_range + J[None, :]).ravel()\n                rows.append(norm_rows)\n                cols.append(np.full(norm_rows.size, start + R))\n                vals.append(np.full(norm_rows.size, -1.0))\n\n                # budget dual\n                rows.append(ub_rows + row0)\n                cols.append(np.full(ub_rows.size, start + R + 1))\n                vals.append(ub_vals)\n\n                # child values\n                child_cols = np.fromiter(\n                    (child_variable(t, j) for j in J), dtype=np.int64,\n                    count=J.size,\n                )\n                rows.append(norm_rows)\n                cols.append(np.tile(child_cols, n))\n                vals.append(np.ones(norm_rows.size))\n\n                # node_value >= normalisation dual:  e[R] - v <= 0\n                v_rows.append(p)\n                v_cols.append((start + R, start + R + 2))\n                v_vals.append((1.0, -1.0))\n\n        main_rows = n_prefix * (nm + 1)\n        dual = coo_matrix(\n            (np.concatenate(vals),\n             (np.concatenate(rows), np.concatenate(cols))),\n            shape=(main_rows, n_vars),\n        ).tocsr()\n\n        node = coo_matrix(\n            (\n                np.array(v_vals).ravel(),\n                (np.repeat(v_rows, 2), np.array(v_cols).ravel()),\n            ),\n            shape=(n_prefix, n_vars),\n        ).tocsr()\n\n        gain = hstack(\n            [block_gain, csr_matrix((m, n_vars - block_gain.shape[1]))],\n            format="csr",\n        )\n\n        A = vstack([gain, dual, node], format="csr")\n        b = np.concatenate([-payoffs, np.zeros(main_rows + n_prefix)])\n\n        lower = np.full(n_vars, -np.inf)\n        upper = np.full(n_vars, np.inf)\n        lower[0], upper[0] = bounds[0]\n        lower[1:1 + nh], upper[1:1 + nh] = bounds[1]\n        lower[i_s:i_s + m] = 0.0\n        for t in range(n_times - 1):\n            for start in group_start[t]:\n                lower[start + R + 1] = 0.0          # budget dual >= 0\n\n        root_value = group_start[0][0] + R + 2\n        c = np.zeros(n_vars)\n        c[0] = 1.0\n        c[root_value] = self.penalty\n\n        res = self._solve_lp(c, A, b, lower, upper, "Rectangular")\n        return self._result(\n            res.x, nh, m, histories_by_time, offsets,\n            res.fun, res.x[root_value],\n        )\n\n    def _rectangular_zero_budget(\n        self, transport_model, bounds, paths, payoffs, block_gain,\n        nh, m, histories_by_time, offsets,\n    ):\n        lookup = {path.tobytes(): j for j, path in enumerate(paths)}\n        weights = np.zeros(m)\n        for path, probability in zip(\n            transport_model.reference_paths,\n            transport_model.reference_probabilities,\n        ):\n            weights[lookup[np.ascontiguousarray(path).tobytes()]] += probability\n\n        c = np.concatenate([[1.0], np.zeros(nh), self.penalty * weights])\n        lower = np.concatenate([[bounds[0, 0]], np.full(nh, bounds[1, 0]), np.zeros(m)])\n        upper = np.concatenate([[bounds[0, 1]], np.full(nh, bounds[1, 1]), np.full(m, np.inf)])\n\n        res = self._solve_lp(c, block_gain, -payoffs, lower, upper, "Rectangular (zero budget)")\n        shortfall = weights @ res.x[1 + nh:]\n        return self._result(\n            res.x, nh, m, histories_by_time, offsets, res.fun, shortfall\n        )\n'
fast_module = types.ModuleType('affine_full_coupling_fast')
exec(compile(FAST_SOURCE, '<embedded full-coupling fast solver>', 'exec'), fast_module.__dict__)
FastRectangularSolver = fast_module.FastRectangularSolver
FastRelaxedSuperhedgingSolver = fast_module.FastRelaxedSuperhedgingSolver


In [ ]:
TRANSPORT_SOURCE = 'class FullCouplingTransport(FastRectangularSolver):\n    """Finite reference tree and full Cartesian adversarial path support.\n\n    No nearest-neighbor coupling restriction. The only numerical support\n    restriction is the finite set of reference states available at each date.\n    """\n    def __init__(self, model, trading_times, budget, discretization):\n        self.reference_model = model\n        self.trading_times = np.asarray(trading_times, dtype=float)\n        self.budget = float(budget)\n        self.discretization = dict(discretization)\n        self.state_space = model.is_valid_state\n        if (self.trading_times.ndim != 1 or len(self.trading_times) < 2\n                or self.trading_times[0] != 0 or np.any(np.diff(self.trading_times) <= 0)\n                or not np.isfinite(self.trading_times).all()):\n            raise ValueError(\'Invalid trading times.\')\n        B = discretization[\'reference_branches\']\n        if isinstance(B, bool) or not isinstance(B, (int, np.integer)) or B < 1:\n            raise ValueError(\'reference_branches must be a positive integer.\')\n        rng = np.random.default_rng(discretization[\'seed\'])\n        initial = np.array([model.s0, model.v0], dtype=float)\n        nodes = [dict(state=initial, children=[], probabilities=np.array([]), time_index=0)]\n        paths = [initial[None, :]]\n        level = [0]\n        for t, dt in enumerate(np.diff(self.trading_times)):\n            next_level, next_paths = [], []\n            for parent_id, history in zip(level, paths):\n                parent = nodes[parent_id]\n                states = np.asarray(model.step(np.tile(parent[\'state\'], (B, 1)), dt, rng))\n                if states.shape != (B, 2) or not np.isfinite(states).all():\n                    raise ValueError(\'Invalid Heston transition output.\')\n                if not all(self.state_space(s) for s in states):\n                    raise ValueError(\'Inadmissible Heston transition output.\')\n                parent[\'probabilities\'] = np.full(B, 1/B)\n                for state in states:\n                    child = len(nodes)\n                    parent[\'children\'].append(child)\n                    nodes.append(dict(state=state.copy(), children=[],\n                                      probabilities=np.array([]), time_index=t+1))\n                    next_level.append(child)\n                    next_paths.append(np.vstack([history, state]))\n            level, paths = next_level, next_paths\n        self.reference_nodes = nodes\n        self.reference_paths = np.asarray(paths, dtype=float)\n        self.reference_probabilities = np.full(len(paths), 1/len(paths))\n        self.adversarial_candidates = {\n            t: np.unique(self.reference_paths[:, t, :], axis=0)\n            for t in range(len(self.trading_times))\n        }\n        self.adversarial_paths = self._build_adversarial_paths()\n        self.transport_costs = self._build_transport_costs()\n        n, m = self.transport_costs.shape\n        reference_A = scipy.sparse.kron(scipy.sparse.eye(n), np.ones((1, m)), format=\'csr\')\n        causal_A, causal_b = self._build_causality_constraints()\n        self.A_eq = vstack([reference_A, causal_A], format=\'csr\')\n        self.b_eq = np.concatenate([self.reference_probabilities, causal_b])\n        self.A_ub = csr_matrix(self.transport_costs.reshape(1, -1))\n        self.set_budget(budget)\n\n    def set_budget(self, budget):\n        if not np.isfinite(budget) or budget < 0:\n            raise ValueError(\'Budget must be finite and nonnegative.\')\n        self.budget = float(budget)\n        self.b_ub = np.array([self.budget**2])\n'
exec(compile(TRANSPORT_SOURCE, '<full coupling transport>', 'exec'))


In [ ]:
CORE_SOURCE = 'def affine_features(paths, s0, v0):\n    paths = np.asarray(paths, dtype=float)\n    stock, variance = paths[:, :-1, 0], paths[:, :-1, 1]\n    return np.stack([np.ones_like(stock), stock / s0 - 1.0, variance - v0], axis=2)\n\n\ndef affine_gains_matrix(paths, s0, v0):\n    features = affine_features(paths, s0, v0)\n    return (features * np.diff(paths[:, :, 0], axis=1)[:, :, None]).reshape(len(paths), -1)\n\n\ndef holding_design(paths, s0, v0):\n    # One row per distinct observed market state at each trading date.\n    T = paths.shape[1] - 1\n    rows = []\n    for t in range(T):\n        market_states = np.unique(paths[:, t, :], axis=0)\n        block = np.zeros((len(market_states), 3 * T))\n        block[:, 3*t:3*t+3] = np.column_stack([\n            np.ones(len(market_states)),\n            market_states[:, 0] / s0 - 1.0,\n            market_states[:, 1] - v0,\n        ])\n        rows.append(block)\n    return csr_matrix(np.vstack(rows))\n\n\ndef gain_path_jacobian(paths, s0, v0):\n    """D[i] @ theta = full gradient of trading gains at reference path i.\n\n    Perturbable coordinates are (S1,V1,...,ST,VT); initial state is fixed.\n    Includes both H_t*dDeltaS_t and DeltaS_t*dH_t.\n    """\n    paths = np.asarray(paths, dtype=float)\n    n, n_times, _ = paths.shape\n    T = n_times - 1\n    features = affine_features(paths, s0, v0)\n    ds = np.diff(paths[:, :, 0], axis=1)\n    D = np.zeros((n, 2*T, 3*T))\n    for t in range(T):\n        block = slice(3*t, 3*t+3)\n        # Derivative through next stock in DeltaS_t.\n        D[:, 2*t, block] += features[:, t]\n        if t > 0:\n            # Derivative through current stock in DeltaS_t.\n            D[:, 2*(t-1), block] -= features[:, t]\n            # Derivatives through the adaptive holding itself.\n            D[:, 2*(t-1), 3*t+1] += ds[:, t] / s0\n            D[:, 2*(t-1)+1, 3*t+2] += ds[:, t]\n    return D\n\n\nclass AffineFastOuter(FastRelaxedSuperhedgingSolver):\n    """Same finite transport LPs, with a common affine hedge parameterization."""\n    def __init__(self, problem, inner_solver, penalty, s0, v0, holding_bounds):\n        super().__init__(problem=problem, inner_solver=inner_solver, penalty=penalty)\n        self.s0, self.v0 = float(s0), float(v0)\n        self.actual_holding_bounds = tuple(holding_bounds)\n\n    def _build_adaptive_gains_matrix(self, paths):\n        T = paths.shape[1] - 1\n        self.F = holding_design(paths, self.s0, self.v0)\n        self.n_coefficients = 3*T\n        G = csr_matrix(affine_gains_matrix(paths, self.s0, self.v0))\n        histories = [np.unique(paths[:, :t+1, :], axis=0) for t in range(T)]\n        return G, histories, np.arange(T) * 3\n\n    def _solve_lp(self, c, A, b, lower, upper, label):\n        # The inherited LP bounds its \'holding variables\'. In this subclass\n        # those variables are affine coefficients; actual holdings are F theta.\n        n = self.n_coefficients\n        F_full = hstack([\n            csr_matrix((self.F.shape[0], 1)), self.F,\n            csr_matrix((self.F.shape[0], len(c)-1-n)),\n        ], format=\'csr\')\n        lo, hi = self.actual_holding_bounds\n        A = vstack([A, F_full, -F_full], format=\'csr\')\n        b = np.concatenate([b, np.full(self.F.shape[0], hi), np.full(self.F.shape[0], -lo)])\n        lower, upper = lower.copy(), upper.copy()\n        # Initial stock/variance are fixed, so these two coefficients are redundant.\n        lower[2:4] = 0.0\n        upper[2:4] = 0.0\n        return fast_module._solve_lp(c, A, b, lower, upper, label)\n\n\ndef solve_first_order_model(tm, penalty, capital_bounds, coefficient_bounds, holding_bounds, strike):\n    """Optimized ordinary-W2 DRO of the first-order trading-gain model.\n\n    The call and shortfall max operators are retained exactly. No derivative\n    convention is needed at their kinks. Reference atoms can split their mass.\n    Target perturbations range over R^(2T), with the initial state fixed.\n    """\n    reference = tm.reference_paths\n    p = np.asarray(tm.reference_probabilities, dtype=float)\n    s0 = float(reference[0, 0, 0])\n    v0 = float(reference[0, 0, 1])\n    G = affine_gains_matrix(reference, s0, v0)\n    D = gain_path_jacobian(reference, s0, v0)\n    F = holding_design(tm.adversarial_paths, s0, v0)\n    n, dimension, nc = D.shape\n    x = cp.Variable()\n    theta = cp.Variable(nc)\n    z = cp.Variable(n, nonneg=True)\n    constraints = [\n        x >= capital_bounds[0], x <= capital_bounds[1],\n        theta >= coefficient_bounds[0], theta <= coefficient_bounds[1],\n        theta[1:3] == 0,\n        F @ theta >= holding_bounds[0], F @ theta <= holding_bounds[1],\n    ]\n    # call(S_T) - wealth, followed by positive part, equals\n    # max(0, -wealth, S_T-K-wealth). Keep both affine payoff branches.\n    a0 = -x - G @ theta\n    a1 = reference[:, -1, 0] - strike + a0\n    if tm.budget == 0:\n        constraints += [z >= a0, z >= a1]\n        robust = p @ z\n        problem = cp.Problem(cp.Minimize(x + penalty * robust), constraints)\n        problem.solve(solver=\'CLARABEL\', verbose=False,\n                      tol_gap_abs=1e-9, tol_gap_rel=1e-9, tol_feas=1e-9)\n        multiplier = None\n    else:\n        lam = cp.Variable(nonneg=True)\n        terminal_direction = np.zeros(dimension)\n        terminal_direction[-2] = 1.0\n        for i in range(n):\n            g0 = -D[i] @ theta\n            for intercept, gradient in [(a0[i], g0), (a1[i], g0 + terminal_direction)]:\n                q = z[i] - intercept\n                # ||(gradient, q-lambda)|| <= q+lambda\n                # is equivalent to ||gradient||^2 <= 4 lambda q.\n                constraints.append(cp.SOC(q + lam, cp.hstack([gradient, q - lam])))\n        robust = tm.budget**2 * lam + p @ z\n        problem = cp.Problem(cp.Minimize(x + penalty * robust), constraints)\n        problem.solve(solver=\'CLARABEL\', verbose=False,\n                      tol_gap_abs=1e-8, tol_gap_rel=1e-8, tol_feas=1e-8,\n                      max_iter=500)\n        multiplier = None if lam.value is None else float(lam.value)\n    if problem.status != cp.OPTIMAL:\n        raise RuntimeError(f\'First-order SOCP failed: {problem.status}\')\n    violations = [np.max(np.asarray(c.violation())) for c in constraints]\n    max_violation = float(max(violations))\n    if not np.isfinite(problem.value) or max_violation > 1e-5:\n        raise RuntimeError(f\'First-order feasibility check failed: {max_violation}\')\n    return {\n        \'capital\': float(x.value), \'coefficients\': np.asarray(theta.value).ravel(),\n        \'value\': float(problem.value), \'worst_case_shortfall\': float(robust.value),\n        \'multiplier\': multiplier, \'max_constraint_violation\': max_violation,\n    }\n\n\ndef gain_remainder_matrix(theta, T, s0):\n    """Exact quadratic remainder: G(z+delta)-G(z)-DG(z)delta = delta.T Q delta."""\n    Q = np.zeros((2*T, 2*T))\n    theta = np.asarray(theta).reshape(T, 3)\n    for t in range(1, T):\n        k, j, l = 2*(t-1), 2*t, 2*(t-1)+1\n        b, c = theta[t, 1] / s0, theta[t, 2]\n        Q[k, k] -= b\n        Q[k, j] += b/2; Q[j, k] += b/2\n        Q[l, j] += c/2; Q[j, l] += c/2\n        Q[l, k] -= c/2; Q[k, l] -= c/2\n    return Q\n'
exec(compile(CORE_SOURCE, '<affine hedging core>', 'exec'))


## Configuration and local saving

The defaults run 20 paired seeds, 1–4 trading intervals, 20 budgets, and 30 penalties. Maturity stays at one year; the budget/penalty sweeps use three intervals. Start with fewer seeds if you need a runtime pilot, then restore the seed list and rerun: successful checkpoints are reused. Four-interval full-coupling rectangular LPs can remain expensive; no neighbor shortcut is applied. Do not increase the branch count casually: the full path product grows rapidly.

Inspect `OUTPUT_DIR` in the variable explorer to find the local results. Optimization errors stop the sweep and save an error record. Means are produced only when every requested seed succeeded. No error bars are plotted, as requested; per-seed values are preserved.


In [ ]:
MODEL_PARAMETERS = dict(s0=100.0, v0=0.04, mu=0.0, kappa=2.0,
                        theta=0.04, vol_of_vol=0.3, rho=-0.7)
SEEDS = list(range(1000, 1020))
STEPS = [1, 2, 3, 4]
BUDGETS = np.linspace(0.0, 2.0, 20).tolist()
PENALTIES = np.geomspace(1.0, 50.0, 30).tolist()
SWEEP_STEPS, FIXED_BUDGET, FIXED_PENALTY = 3, 1.0, 10.0
MATURITY, STRIKE, BRANCHES = 1.0, 100.0, 2
CAPITAL_BOUNDS, HOLDING_BOUNDS = (0.0, 50.0), (-2.0, 2.0)
COEFFICIENT_BOUNDS = (-2.0, 2.0)
RESULTS_ROOT = Path.home() / 'awdro_results'


def call_payoff(paths):
    return torch.clamp(paths[:, -1, 0] - STRIKE, min=0.0)


def save_json(path, content):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(content, indent=2, allow_nan=False))
    temporary.replace(path)


# The source hashes prevent silently reusing results from a different solver.
base_sources = {}
for cls in [Heston, SuperhedgingProblem, RectangularSolver, RelaxedSuperhedgingSolver]:
    source = Path(inspect.getfile(cls))
    base_sources[cls.__name__] = hashlib.sha256(source.read_bytes()).hexdigest()
config = dict(
    version='affine-full-derivative-v1', model=MODEL_PARAMETERS,
    maturity=MATURITY, strike=STRIKE, branches=BRANCHES,
    capital_bounds=CAPITAL_BOUNDS, holding_bounds=HOLDING_BOUNDS,
    coefficient_bounds=COEFFICIENT_BOUNDS, base_source_hashes=base_sources,
    fast_source_hash=hashlib.sha256(FAST_SOURCE.encode()).hexdigest(),
    core_source_hash=hashlib.sha256(CORE_SOURCE.encode()).hexdigest(),
    transport_source_hash=hashlib.sha256(TRANSPORT_SOURCE.encode()).hexdigest(),
    cost='sum_t [(S_t-Y_S_t)^2 + (V_t-Y_V_t)^2], fixed initial state; no dt weights',
    finite_support='full Cartesian product of reference states at each date',
    first_order='ordinary W2 DRO of first-order trading gains; exact call/shortfall branches',
    policy='H_t = a_t + b_t*(S_t/S0-1) + c_t*(V_t-V0); a,b,c bounded; b0=c0=0',
    packages={p: importlib.metadata.version(p) for p in
              ['numpy', 'scipy', 'pandas', 'torch', 'cvxpy', 'clarabel', 'matplotlib']},
)
STUDY_ID = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
OUTPUT_DIR = RESULTS_ROOT / ('affine_three_sweeps_' + STUDY_ID)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_json(OUTPUT_DIR / 'configuration.json', config)
save_json(OUTPUT_DIR / 'requested_runs.json', dict(seeds=SEEDS, steps=STEPS,
          budgets=BUDGETS, penalties=PENALTIES, sweep_steps=SWEEP_STEPS,
          fixed_budget=FIXED_BUDGET, fixed_penalty=FIXED_PENALTY))
(OUTPUT_DIR / 'embedded_fast_source.py').write_text(FAST_SOURCE)
(OUTPUT_DIR / 'affine_core.py').write_text(CORE_SOURCE)
(OUTPUT_DIR / 'transport.py').write_text(TRANSPORT_SOURCE)


# Only one tree is retained in memory at a time. Its support is reused across
# budgets and penalties; every method uses exactly the same reference sample.
_TREE_CACHE = {}

def get_transport(seed, n_steps):
    key = (int(seed), int(n_steps))
    if key not in _TREE_CACHE:
        _TREE_CACHE.clear()
        model = Heston(**MODEL_PARAMETERS)
        times = np.linspace(0.0, MATURITY, int(n_steps)+1)
        tm = FullCouplingTransport(model, times, 0.0,
              dict(seed=int(seed), reference_branches=BRANCHES))
        problem = SuperhedgingProblem(payoff=call_payoff, trading_times=times)
        _TREE_CACHE[key] = (tm, problem)
    return _TREE_CACHE[key]


def check_result(result, tm, penalty):
    theta = np.asarray(result['coefficients'])
    x, value, short = [float(result[k]) for k in ['capital', 'value', 'worst_case_shortfall']]
    if not np.isfinite([x, value, short]).all() or not np.isfinite(theta).all():
        raise RuntimeError('Nonfinite optimization result.')
    tol = 1e-5 * max(1.0, abs(value))
    np.testing.assert_allclose(value, x + penalty*short, atol=tol, rtol=0)
    if short < -tol or not CAPITAL_BOUNDS[0]-tol <= x <= CAPITAL_BOUNDS[1]+tol:
        raise RuntimeError('Capital/shortfall feasibility failure.')
    if (theta < COEFFICIENT_BOUNDS[0]-tol).any() or (theta > COEFFICIENT_BOUNDS[1]+tol).any():
        raise RuntimeError('Coefficient bounds violated.')
    h = holding_design(tm.adversarial_paths, MODEL_PARAMETERS['s0'], MODEL_PARAMETERS['v0']) @ theta
    if (h < HOLDING_BOUNDS[0]-tol).any() or (h > HOLDING_BOUNDS[1]+tol).any():
        raise RuntimeError('Holding bounds violated.')


def run_point(seed, n_steps, budget, penalty):
    spec = dict(seed=int(seed), n_steps=int(n_steps), budget=float(budget), penalty=float(penalty))
    key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:20]
    checkpoint = OUTPUT_DIR / (key + '.json')
    if checkpoint.exists():
        saved = json.loads(checkpoint.read_text())
        if saved.get('status') == 'ok':
            return saved
    started = time.perf_counter()
    try:
        tm, problem = get_transport(seed, n_steps)
        tm.set_budget(budget)
        outer = AffineFastOuter(problem, tm, penalty, MODEL_PARAMETERS['s0'],
                                MODEL_PARAMETERS['v0'], HOLDING_BOUNDS)
        results = {}
        for name, method in [
            ('nonrectangular', outer.solve_adaptive_hedge_nonrectangular),
            ('rectangular', outer.solve_adaptive_hedge_rectangular),
        ]:
            result = method(tm, CAPITAL_BOUNDS, COEFFICIENT_BOUNDS)
            result['coefficients'] = result.pop('holdings')
            results[name] = result
        results['first_order'] = solve_first_order_model(
            tm, penalty, CAPITAL_BOUNDS, COEFFICIENT_BOUNDS, HOLDING_BOUNDS, STRIKE)
        for result in results.values():
            check_result(result, tm, penalty)
        nr, rect, fo = [results[k]['value'] for k in ['nonrectangular', 'rectangular', 'first_order']]
        tol = 1e-5 * max(1, abs(nr), abs(rect))
        if rect < nr - tol:
            raise RuntimeError('Negative rectangularity gap beyond tolerance.')
        if budget == 0:
            np.testing.assert_allclose([rect, fo], nr, rtol=1e-5, atol=1e-5)
        if n_steps == 1:
            np.testing.assert_allclose(rect, nr, rtol=1e-5, atol=1e-5)
        row = dict(**spec, status='ok', seconds=time.perf_counter()-started)
        arrays = dict(reference_paths=tm.reference_paths,
                      reference_probabilities=tm.reference_probabilities,
                      adversarial_paths=tm.adversarial_paths,
                      trading_times=tm.trading_times)
        for name, result in results.items():
            for field in ['value', 'capital', 'worst_case_shortfall']:
                row[name + '_' + field] = float(result[field])
            arrays[name + '_coefficients'] = result['coefficients'].reshape(n_steps, 3)
        row['gap'] = rect - nr
        row['first_order_fixed_policy_remainder_bound'] = float(
            penalty * budget**2 * np.linalg.norm(gain_remainder_matrix(
                results['first_order']['coefficients'], n_steps, MODEL_PARAMETERS['s0']), ord=2))
        row['first_order_max_constraint_violation'] = results['first_order']['max_constraint_violation']
        np.savez_compressed(OUTPUT_DIR / (key + '_policies.npz'), **arrays)
        save_json(checkpoint, row)
        return row
    except Exception as exc:
        save_json(OUTPUT_DIR / (key + '_error.json'),
                  dict(**spec, error=repr(exc), seconds=time.perf_counter()-started))
        # Never average only the seeds whose optimizations happened to succeed.
        raise


def run_sweep(name, xs, specifications):
    rows = []
    for seed in SEEDS:
        for value in xs:
            n_steps, budget, penalty = specifications(value)
            row = dict(run_point(seed, n_steps, budget, penalty), sweep_x=float(value))
            rows.append(row)
            pd.DataFrame(rows).to_csv(OUTPUT_DIR / (name + '_partial.csv'), index=False)
    raw = pd.DataFrame(rows)
    if raw.duplicated(['seed', 'sweep_x']).any():
        raise RuntimeError('Duplicate experiment row.')
    if not (raw.groupby('sweep_x').size() == len(SEEDS)).all():
        raise RuntimeError('Unequal seed counts.')
    columns = [k + '_value' for k in ['nonrectangular', 'rectangular', 'first_order']]
    means = raw.groupby('sweep_x', as_index=False)[columns].mean()
    raw.to_csv(OUTPUT_DIR / (name + '_by_seed.csv'), index=False)
    means.to_csv(OUTPUT_DIR / (name + '_means.csv'), index=False)
    return raw, means


def plot_means(means, xlabel, filename, logarithmic=False, output_dir=None):
    destination = OUTPUT_DIR if output_dir is None else Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    labels = dict(nonrectangular='Nonrectangular causal', rectangular='Rectangular hull',
                  first_order='First-order ordinary W₂')
    colors = dict(nonrectangular='#633A86', rectangular='#ECA477', first_order='#B99BCF')
    with plt.rc_context({'font.size': 11, 'axes.spines.top': False,
                         'axes.spines.right': False, 'axes.edgecolor': '#BDB7C2',
                         'xtick.color': '#514957', 'ytick.color': '#514957',
                         'axes.labelcolor': '#514957', 'figure.facecolor': 'white',
                         'axes.facecolor': 'white', 'savefig.facecolor': 'white'}):
        fig, ax = plt.subplots(figsize=(6.2, 4.0), layout='constrained')
        for name in ['nonrectangular', 'rectangular', 'first_order']:
            ax.plot(means.sweep_x, means[name+'_value'], color=colors[name], lw=2,
                    linestyle='-', label=labels[name],
                    marker='o', markersize=3.2)
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Mean optimized objective')
        if logarithmic:
            ax.set_xscale('log')
        if filename == 'time_steps':
            ax.set_xticks(STEPS)
        ax.grid(axis='both', which='major', color='#EEEAF0', linewidth=0.65)
        ax.legend(loc='best', frameon=True, facecolor='white', edgecolor='none',
                  framealpha=0.9, fontsize=9)
        ax.set_axisbelow(True)
        for extension in ['pdf', 'png']:
            fig.savefig(destination / (filename + '.' + extension), dpi=300)
        plt.show()
        plt.close(fig)


In [ ]:
# These checks exercise the derivative, its quadratic remainder, and a kink.
# They do not replace convergence tests for the finite reference model.
rng = np.random.default_rng(79)
test_paths = np.array([[[100., .04], [104., .05], [98., .03], [108., .06]]])
test_theta = np.array([.3, 0., 0., .2, .7, -.4, -.1, -.5, .8])
D = gain_path_jacobian(test_paths, 100., .04)[0]
analytic = D @ test_theta
finite_difference = np.zeros(6)
for j in range(6):
    plus, minus = test_paths.copy(), test_paths.copy()
    step = 1e-5
    plus[0, 1+j//2, j%2] += step
    minus[0, 1+j//2, j%2] -= step
    finite_difference[j] = ((affine_gains_matrix(plus, 100., .04)
                            - affine_gains_matrix(minus, 100., .04)) @ test_theta)[0] / (2*step)
np.testing.assert_allclose(analytic, finite_difference, atol=1e-8, rtol=1e-7)

delta = rng.normal(size=6) * .02
perturbed = test_paths.copy()
perturbed[0, 1:, :] += delta.reshape(3, 2)
remainder = ((affine_gains_matrix(perturbed, 100., .04)
              - affine_gains_matrix(test_paths, 100., .04)) @ test_theta)[0] - analytic @ delta
Q = gain_remainder_matrix(test_theta, 3, 100.)
np.testing.assert_allclose(remainder, delta @ Q @ delta, atol=1e-12)

# One reference atom exactly at the call strike, no hedge, zero capital:
# sup E[(S'-100)+] under E[(S'-100)^2+(V'-.04)^2] <= eps^2 is eps.
# A frozen zero autodiff subgradient would incorrectly give zero.
from types import SimpleNamespace
kink_paths = np.array([[[100., .04], [100., .04]]])
kink_tm = SimpleNamespace(reference_paths=kink_paths, adversarial_paths=kink_paths,
                         reference_probabilities=np.array([1.0]), budget=.1)
kink_result = solve_first_order_model(kink_tm, 1., (0., 0.), (0., 0.), (-2., 2.), 100.)
np.testing.assert_allclose(kink_result['value'], .1, atol=1e-6, rtol=1e-6)


In [ ]:
# Optional: refresh figures from completed CSV files without solving again.
# Run the setup/configuration cells above first. If CSVs are absent, use the
# original sweep cells below. Skip those sweeps when you only want a restyle.
for name, xlabel, logarithmic in [
    ('time_steps', 'Trading intervals', False),
    ('budget', 'Wasserstein radius ε', False),
    ('penalty', 'Penalty κ', True),
]:
    saved_means = OUTPUT_DIR / (name + '_means.csv')
    if saved_means.exists():
        plot_means(pd.read_csv(saved_means), xlabel, name, logarithmic=logarithmic)


In [ ]:
steps_by_seed, steps_means = run_sweep('time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(steps_means, 'Trading intervals', 'time_steps')


In [ ]:
budget_by_seed, budget_means = run_sweep('budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(budget_means, 'Wasserstein radius ε', 'budget')


In [ ]:
penalty_by_seed, penalty_means = run_sweep('penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(penalty_means, 'Penalty κ', 'penalty', logarithmic=True)


## Additional option experiments

The following cells add **European put, straddle, strangle, bull call spread, bear put spread, butterfly, digital call, and digital put**. Each payoff has three separate executable plot cells. They use the same `SEEDS`, `STEPS`, `BUDGETS`, `PENALTIES`, affine hedge bounds and transport convention as the original call experiment. Changing `STRIKE`, `WING` or `DIGITAL_CASH` changes the payoff definitions; it creates separate payoff checkpoints.

The full set is a substantial run: eight more copies of all three sweeps. Run the payoff cells you need; each completed point is saved and reused. All graphs remain title-free; the notebook headings and filenames identify the payoff. Results are under `OUTPUT_DIR / 'payoffs' / '<payoff>_<hash>'`. Digitals pay `DIGITAL_CASH` (default 1), while vanilla/spread/straddle payoffs are in stock-price units; magnitudes across payoff types are not normalized.

**First-order meaning remains unchanged:** first-order trading gains with the exact payoff retained, ordinary W2 and unrestricted endpoint perturbations. For nonconvex payoffs we retain the domain of every affine segment; we do not replace the payoff by a convex envelope. The digital jump is retained, not differentiated or smoothed. For digitals, this is not a classical differentiable Wasserstein sensitivity formula. A reference atom at a digital strike can cause a discontinuity between zero and arbitrarily small positive budgets.

For a segment $f(s)=ms+b$ on $\ell\le s\le u$, set $a=mS_T+b-x-G$, $g=me_{S_T}-DG$, and introduce $\eta,\zeta\ge0$ for the finite upper/lower bounds. Its dual constraint is
$$4\lambda\{z-a-\eta(u-S_T)-\zeta(S_T-\ell)\}\ge
\|g-\eta e_{S_T}+\zeta e_{S_T}\|_2^2.$$
Omit the corresponding multiplier when an endpoint is infinite. This is a second-order cone constraint. At positive budgets the digital segment closures give the supremum; at zero budget the exact strict digital convention is used: call pays on $S_T>K$, put on $S_T<K$, both pay zero at equality.


In [ ]:
PAYOFF_SOURCE = 'def make_payoff_catalog(strike=100.0, wing=10.0, digital_cash=1.0):\n    """Terminal piecewise-affine options. All prices/strikes are in stock units.\n\n    Each segment is (lower endpoint, upper endpoint, slope, intercept).\n    None denotes an infinite endpoint. Digital equality conventions are\n    specified separately; their positive-budget suprema use interval closures.\n    """\n    if not np.isfinite([strike, wing, digital_cash]).all() or wing <= 0 or digital_cash <= 0:\n        raise ValueError(\'Strike must be finite; wing and digital cash positive and finite.\')\n    lo, hi = strike - wing, strike + wing\n    def spec(name, label, segments, discontinuous=False):\n        return dict(name=name, label=label, strike=float(strike), wing=float(wing),\n                    cash=float(digital_cash), segments=segments, discontinuous=discontinuous)\n    return {\n        \'european_call\': spec(\'european_call\', \'European call\',\n            [(None, strike, 0., 0.), (strike, None, 1., -strike)]),\n        \'european_put\': spec(\'european_put\', \'European put\',\n            [(None, strike, -1., strike), (strike, None, 0., 0.)]),\n        \'straddle\': spec(\'straddle\', \'Straddle\',\n            [(None, strike, -1., strike), (strike, None, 1., -strike)]),\n        \'strangle\': spec(\'strangle\', \'Strangle\',\n            [(None, lo, -1., lo), (lo, hi, 0., 0.), (hi, None, 1., -hi)]),\n        \'bull_call_spread\': spec(\'bull_call_spread\', \'Bull call spread\',\n            [(None, lo, 0., 0.), (lo, hi, 1., -lo), (hi, None, 0., hi-lo)]),\n        \'bear_put_spread\': spec(\'bear_put_spread\', \'Bear put spread\',\n            [(None, lo, 0., hi-lo), (lo, hi, -1., hi), (hi, None, 0., 0.)]),\n        \'butterfly\': spec(\'butterfly\', \'Butterfly\',\n            [(None, lo, 0., 0.), (lo, strike, 1., -lo),\n             (strike, hi, -1., hi), (hi, None, 0., 0.)]),\n        \'digital_call\': spec(\'digital_call\', \'Digital call\',\n            [(None, strike, 0., 0.), (strike, None, 0., digital_cash)], True),\n        \'digital_put\': spec(\'digital_put\', \'Digital put\',\n            [(None, strike, 0., digital_cash), (strike, None, 0., 0.)], True),\n    }\n\n\ndef payoff_numpy(stock, spec):\n    stock = np.asarray(stock, dtype=float)\n    if spec[\'name\'] == \'digital_call\':\n        return spec[\'cash\'] * (stock > spec[\'strike\']).astype(float)\n    if spec[\'name\'] == \'digital_put\':\n        return spec[\'cash\'] * (stock < spec[\'strike\']).astype(float)\n    result = np.zeros_like(stock)\n    for lower, upper, slope, intercept in spec[\'segments\']:\n        mask = np.ones(stock.shape, dtype=bool)\n        if lower is not None:\n            mask &= stock >= lower\n        if upper is not None:\n            mask &= stock < upper\n        result = np.where(mask, slope*stock + intercept, result)\n    return result\n\n\ndef payoff_tensor(paths, spec):\n    stock = paths[:, -1, 0]\n    if spec[\'name\'] == \'digital_call\':\n        return spec[\'cash\'] * (stock > spec[\'strike\']).to(stock.dtype)\n    if spec[\'name\'] == \'digital_put\':\n        return spec[\'cash\'] * (stock < spec[\'strike\']).to(stock.dtype)\n    result = torch.zeros_like(stock)\n    for lower, upper, slope, intercept in spec[\'segments\']:\n        mask = torch.ones_like(stock, dtype=torch.bool)\n        if lower is not None:\n            mask = mask & (stock >= lower)\n        if upper is not None:\n            mask = mask & (stock < upper)\n        result = torch.where(mask, slope*stock + intercept, result)\n    return result\n\n\ndef solve_first_order_payoff(tm, penalty, capital_bounds, coefficient_bounds,\n                             holding_bounds, spec):\n    """First-order trading gains, exact piecewise-affine terminal payoff.\n\n    The segment domain is retained even for nonconvex/discontinuous payoffs.\n    Replacing segments by an unrestricted maximum would incorrectly convexify\n    spreads/butterflies. This routine never differentiates a digital indicator.\n    """\n    # Preserve the original call solver exactly.\n    if spec[\'name\'] == \'european_call\':\n        return solve_first_order_model(tm, penalty, capital_bounds,\n                                       coefficient_bounds, holding_bounds, spec[\'strike\'])\n    ref = tm.reference_paths\n    p = np.asarray(tm.reference_probabilities, dtype=float)\n    s0, v0 = ref[0, 0]\n    G = affine_gains_matrix(ref, s0, v0)\n    D = gain_path_jacobian(ref, s0, v0)\n    F = holding_design(tm.adversarial_paths, s0, v0)\n    n, dimension, nc = D.shape\n    x, theta, z = cp.Variable(), cp.Variable(nc), cp.Variable(n, nonneg=True)\n    constraints = [x >= capital_bounds[0], x <= capital_bounds[1],\n                   theta >= coefficient_bounds[0], theta <= coefficient_bounds[1],\n                   theta[1:3] == 0, F @ theta >= holding_bounds[0],\n                   F @ theta <= holding_bounds[1]]\n    wealth = x + G @ theta\n    if tm.budget == 0:\n        constraints.append(z >= payoff_numpy(ref[:, -1, 0], spec) - wealth)\n        robust = p @ z\n        lam = None\n    else:\n        lam = cp.Variable(nonneg=True)\n        terminal_direction = np.zeros(dimension)\n        terminal_direction[-2] = 1.0\n        for i in range(n):\n            stock = ref[i, -1, 0]\n            for lower, upper, slope, intercept in spec[\'segments\']:\n                a = slope*stock + intercept - wealth[i]\n                gradient = slope*terminal_direction - D[i] @ theta\n                domain_penalty = 0.0\n                # Dualize lower <= stock + delta_terminal <= upper.\n                if upper is not None:\n                    eta = cp.Variable(nonneg=True)\n                    gradient = gradient - eta*terminal_direction\n                    domain_penalty += eta*(upper-stock)\n                if lower is not None:\n                    zeta = cp.Variable(nonneg=True)\n                    gradient = gradient + zeta*terminal_direction\n                    domain_penalty += zeta*(stock-lower)\n                q = z[i] - a - domain_penalty\n                constraints.append(cp.SOC(q + lam, cp.hstack([gradient, q - lam])))\n        robust = tm.budget**2 * lam + p @ z\n    problem = cp.Problem(cp.Minimize(x + penalty*robust), constraints)\n    problem.solve(solver=\'CLARABEL\', verbose=False, tol_gap_abs=1e-8,\n                  tol_gap_rel=1e-8, tol_feas=1e-8, max_iter=500)\n    if problem.status != cp.OPTIMAL:\n        raise RuntimeError(f"{spec[\'name\']} first-order SOCP failed: {problem.status}")\n    violation = float(max(np.max(np.asarray(c.violation())) for c in constraints))\n    if not np.isfinite(problem.value) or violation > 1e-5:\n        raise RuntimeError(f\'First-order feasibility failure: {violation}\')\n    return dict(capital=float(x.value), coefficients=np.asarray(theta.value).ravel(),\n                value=float(problem.value), worst_case_shortfall=float(robust.value),\n                multiplier=None if lam is None else float(lam.value),\n                max_constraint_violation=violation)\n\n\ndef payoff_output_dir(spec):\n    fingerprint = dict(payoff=spec, implementation=\'segment-dual-v1\',\n                       source_hash=hashlib.sha256(PAYOFF_SOURCE.encode()).hexdigest())\n    digest = hashlib.sha256(json.dumps(fingerprint, sort_keys=True).encode()).hexdigest()[:12]\n    directory = OUTPUT_DIR / \'payoffs\' / (spec[\'name\'] + \'_\' + digest)\n    directory.mkdir(parents=True, exist_ok=True)\n    if not (directory / \'payoff.json\').exists():\n        save_json(directory / \'payoff.json\', fingerprint)\n        (directory / \'payoff_extensions.py\').write_text(PAYOFF_SOURCE)\n    return directory\n\n\ndef run_option_point(seed, n_steps, budget, penalty, payoff_spec):\n    directory = payoff_output_dir(payoff_spec)\n    spec = dict(seed=int(seed), n_steps=int(n_steps), budget=float(budget), penalty=float(penalty))\n    key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:20]\n    checkpoint = directory / (key + \'.json\')\n    if checkpoint.exists():\n        saved = json.loads(checkpoint.read_text())\n        if saved.get(\'status\') == \'ok\':\n            return saved\n    started = time.perf_counter()\n    try:\n        tm, problem = get_transport(seed, n_steps)\n        problem = SuperhedgingProblem(\n            payoff=lambda paths: payoff_tensor(paths, payoff_spec), trading_times=tm.trading_times)\n        tm.set_budget(budget)\n        outer = AffineFastOuter(problem, tm, penalty, MODEL_PARAMETERS[\'s0\'],\n                                MODEL_PARAMETERS[\'v0\'], HOLDING_BOUNDS)\n        results = {}\n        for name, method in [\n            (\'nonrectangular\', outer.solve_adaptive_hedge_nonrectangular),\n            (\'rectangular\', outer.solve_adaptive_hedge_rectangular),\n        ]:\n            result = method(tm, CAPITAL_BOUNDS, COEFFICIENT_BOUNDS)\n            result[\'coefficients\'] = result.pop(\'holdings\')\n            results[name] = result\n        results[\'first_order\'] = solve_first_order_payoff(\n            tm, penalty, CAPITAL_BOUNDS, COEFFICIENT_BOUNDS, HOLDING_BOUNDS, payoff_spec)\n        for result in results.values():\n            check_result(result, tm, penalty)\n        nr, rect, fo = [results[k][\'value\'] for k in [\'nonrectangular\', \'rectangular\', \'first_order\']]\n        tol = 1e-5 * max(1, abs(nr), abs(rect))\n        if rect < nr - tol:\n            raise RuntimeError(\'Negative rectangularity gap beyond tolerance.\')\n        if budget == 0:\n            np.testing.assert_allclose([rect, fo], nr, rtol=1e-5, atol=1e-5)\n        if n_steps == 1:\n            np.testing.assert_allclose(rect, nr, rtol=1e-5, atol=1e-5)\n        row = dict(**spec, payoff=payoff_spec[\'name\'], status=\'ok\', seconds=time.perf_counter()-started)\n        arrays = dict(reference_paths=tm.reference_paths,\n                      reference_probabilities=tm.reference_probabilities,\n                      adversarial_paths=tm.adversarial_paths,\n                      trading_times=tm.trading_times)\n        for name, result in results.items():\n            for field in [\'value\', \'capital\', \'worst_case_shortfall\']:\n                row[name + \'_\' + field] = float(result[field])\n            arrays[name + \'_coefficients\'] = result[\'coefficients\'].reshape(n_steps, 3)\n        row[\'gap\'] = rect - nr\n        row[\'first_order_fixed_policy_remainder_bound\'] = float(\n            penalty * budget**2 * np.linalg.norm(gain_remainder_matrix(\n                results[\'first_order\'][\'coefficients\'], n_steps, MODEL_PARAMETERS[\'s0\']), ord=2))\n        row[\'first_order_max_constraint_violation\'] = results[\'first_order\'][\'max_constraint_violation\']\n        np.savez_compressed(directory / (key + \'_policies.npz\'), **arrays)\n        save_json(checkpoint, row)\n        return row\n    except Exception as exc:\n        save_json(directory / (key + \'_error.json\'),\n                  dict(**spec, error=repr(exc), seconds=time.perf_counter()-started))\n        # Never average only the seeds whose optimizations happened to succeed.\n        raise\n\n\n\ndef run_option_sweep(payoff_spec, name, xs, specifications):\n    directory = payoff_output_dir(payoff_spec)\n    save_json(directory / (name + \'_request.json\'), dict(seeds=SEEDS, values=list(xs),\n              maturity=MATURITY, sweep_steps=SWEEP_STEPS, fixed_budget=FIXED_BUDGET,\n              fixed_penalty=FIXED_PENALTY))\n    rows = []\n    for seed in SEEDS:\n        for value in xs:\n            n_steps, budget, penalty = specifications(value)\n            row = dict(run_option_point(seed, n_steps, budget, penalty, payoff_spec), sweep_x=float(value))\n            rows.append(row)\n            pd.DataFrame(rows).to_csv(directory / (name + \'_partial.csv\'), index=False)\n    raw = pd.DataFrame(rows)\n    if raw.duplicated([\'seed\', \'sweep_x\']).any():\n        raise RuntimeError(\'Duplicate experiment row.\')\n    if not (raw.groupby(\'sweep_x\').size() == len(SEEDS)).all():\n        raise RuntimeError(\'Unequal seed counts.\')\n    columns = [k + \'_value\' for k in [\'nonrectangular\', \'rectangular\', \'first_order\']]\n    means = raw.groupby(\'sweep_x\', as_index=False)[columns].mean()\n    raw.to_csv(directory / (name + \'_by_seed.csv\'), index=False)\n    means.to_csv(directory / (name + \'_means.csv\'), index=False)\n    return raw, means\n\n'
exec(compile(PAYOFF_SOURCE, '<piecewise payoff extensions>', 'exec'))


In [ ]:
WING = 10.0  # low/high strikes = STRIKE - WING and STRIKE + WING
DIGITAL_CASH = 1.0
PAYOFFS = make_payoff_catalog(STRIKE, WING, DIGITAL_CASH)

# The call in PAYOFFS is available for validation; the original call experiment
# above continues to use its original first-order solver and cached results.


In [ ]:
# Payoff definition checks at/around every strike.
_check_stock = np.array([STRIKE-2*WING, STRIKE-WING, STRIKE,
                         STRIKE+WING, STRIKE+2*WING])
np.testing.assert_allclose(payoff_numpy(_check_stock, PAYOFFS['butterfly']),
                           [0, 0, WING, 0, 0])
np.testing.assert_allclose(payoff_numpy(_check_stock, PAYOFFS['bull_call_spread']),
                           [0, 0, WING, 2*WING, 2*WING])
np.testing.assert_allclose(payoff_numpy(_check_stock, PAYOFFS['digital_call']),
                           [0, 0, 0, DIGITAL_CASH, DIGITAL_CASH])
_check_paths = np.zeros((len(_check_stock), 2, 2))
_check_paths[:, -1, 0] = _check_stock
for _spec in PAYOFFS.values():
    np.testing.assert_allclose(
        payoff_tensor(torch.tensor(_check_paths, dtype=torch.float64), _spec).numpy(),
        payoff_numpy(_check_stock, _spec), atol=1e-12)


### European put

In [ ]:
european_put_steps_raw, european_put_steps_mean = run_option_sweep(
    PAYOFFS['european_put'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(european_put_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['european_put']))


In [ ]:
european_put_budget_raw, european_put_budget_mean = run_option_sweep(
    PAYOFFS['european_put'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(european_put_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['european_put']))


In [ ]:
european_put_penalty_raw, european_put_penalty_mean = run_option_sweep(
    PAYOFFS['european_put'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(european_put_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['european_put']))


### Straddle

In [ ]:
straddle_steps_raw, straddle_steps_mean = run_option_sweep(
    PAYOFFS['straddle'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(straddle_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['straddle']))


In [ ]:
straddle_budget_raw, straddle_budget_mean = run_option_sweep(
    PAYOFFS['straddle'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(straddle_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['straddle']))


In [ ]:
straddle_penalty_raw, straddle_penalty_mean = run_option_sweep(
    PAYOFFS['straddle'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(straddle_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['straddle']))


### Strangle

In [ ]:
strangle_steps_raw, strangle_steps_mean = run_option_sweep(
    PAYOFFS['strangle'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(strangle_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['strangle']))


In [ ]:
strangle_budget_raw, strangle_budget_mean = run_option_sweep(
    PAYOFFS['strangle'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(strangle_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['strangle']))


In [ ]:
strangle_penalty_raw, strangle_penalty_mean = run_option_sweep(
    PAYOFFS['strangle'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(strangle_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['strangle']))


### Bull call spread

In [ ]:
bull_call_spread_steps_raw, bull_call_spread_steps_mean = run_option_sweep(
    PAYOFFS['bull_call_spread'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(bull_call_spread_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['bull_call_spread']))


In [ ]:
bull_call_spread_budget_raw, bull_call_spread_budget_mean = run_option_sweep(
    PAYOFFS['bull_call_spread'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(bull_call_spread_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['bull_call_spread']))


In [ ]:
bull_call_spread_penalty_raw, bull_call_spread_penalty_mean = run_option_sweep(
    PAYOFFS['bull_call_spread'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(bull_call_spread_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['bull_call_spread']))


### Bear put spread

In [ ]:
bear_put_spread_steps_raw, bear_put_spread_steps_mean = run_option_sweep(
    PAYOFFS['bear_put_spread'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(bear_put_spread_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['bear_put_spread']))


In [ ]:
bear_put_spread_budget_raw, bear_put_spread_budget_mean = run_option_sweep(
    PAYOFFS['bear_put_spread'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(bear_put_spread_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['bear_put_spread']))


In [ ]:
bear_put_spread_penalty_raw, bear_put_spread_penalty_mean = run_option_sweep(
    PAYOFFS['bear_put_spread'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(bear_put_spread_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['bear_put_spread']))


### Butterfly

In [ ]:
butterfly_steps_raw, butterfly_steps_mean = run_option_sweep(
    PAYOFFS['butterfly'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(butterfly_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['butterfly']))


In [ ]:
butterfly_budget_raw, butterfly_budget_mean = run_option_sweep(
    PAYOFFS['butterfly'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(butterfly_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['butterfly']))


In [ ]:
butterfly_penalty_raw, butterfly_penalty_mean = run_option_sweep(
    PAYOFFS['butterfly'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(butterfly_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['butterfly']))


### Digital call

In [ ]:
digital_call_steps_raw, digital_call_steps_mean = run_option_sweep(
    PAYOFFS['digital_call'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(digital_call_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['digital_call']))


In [ ]:
digital_call_budget_raw, digital_call_budget_mean = run_option_sweep(
    PAYOFFS['digital_call'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(digital_call_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['digital_call']))


In [ ]:
digital_call_penalty_raw, digital_call_penalty_mean = run_option_sweep(
    PAYOFFS['digital_call'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(digital_call_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['digital_call']))


### Digital put

In [ ]:
digital_put_steps_raw, digital_put_steps_mean = run_option_sweep(
    PAYOFFS['digital_put'], 'time_steps', STEPS,
    lambda t: (int(t), FIXED_BUDGET, FIXED_PENALTY))
plot_means(digital_put_steps_mean, 'Trading intervals', 'time_steps',
           output_dir=payoff_output_dir(PAYOFFS['digital_put']))


In [ ]:
digital_put_budget_raw, digital_put_budget_mean = run_option_sweep(
    PAYOFFS['digital_put'], 'budget', BUDGETS,
    lambda budget: (SWEEP_STEPS, float(budget), FIXED_PENALTY))
plot_means(digital_put_budget_mean, 'Wasserstein radius ε', 'budget',
           output_dir=payoff_output_dir(PAYOFFS['digital_put']))


In [ ]:
digital_put_penalty_raw, digital_put_penalty_mean = run_option_sweep(
    PAYOFFS['digital_put'], 'penalty', PENALTIES,
    lambda penalty: (SWEEP_STEPS, FIXED_BUDGET, float(penalty)))
plot_means(digital_put_penalty_mean, 'Penalty κ', 'penalty', logarithmic=True,
           output_dir=payoff_output_dir(PAYOFFS['digital_put']))
